In [0]:
table_name = "dim_service_types"
silver_table_fqn = f"hdfc_ergo.hdfc_silver.{table_name}"

df = spark.read.table("hdfc_ergo.hdfc_bronze.dim_service_types")
display(df.limit(5))

In [0]:
from pyspark.sql.functions import upper, trim, col, when, abs as spark_abs

# 1. Standardize text: UPPER(TRIM()) for ID and text columns
# NOTE: service_type_id is 'ST_001', so we keep it as STRING (Business Key). We do NOT cast to INT.
df = df.withColumn("service_type_id", upper(trim(col("service_type_id"))))
df = df.withColumn("service_type_name", upper(trim(col("service_type_name"))))
df = df.withColumn("service_category", upper(trim(col("service_category"))))

# 2. Boolean Optimization for requires_prior_auth
df = df.withColumn("requires_prior_auth", 
                   when(upper(trim(col("requires_prior_auth"))).isin("YES", "1"), True)
                   .when(upper(trim(col("requires_prior_auth"))).isin("NO", "0"), False)
                   .otherwise(None).cast("boolean"))

# 3. Precision Optimization: Safe Cast to DECIMAL(15,2). If < 0, apply ABS()
df = df.withColumn("reference_price", trim(col("reference_price")).cast("decimal(15,2)"))
df = df.withColumn("reference_price", spark_abs(col("reference_price")))

# 4. Drop audit columns and source surrogate key
df = df.drop("_ingested_at", "_source_file", "service_type_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(silver_table_fqn)

print(f"✅ Successfully wrote {table_name} to Silver layer.")